# Text Precision: Taxonomy Mapping & VADER Sentiment Analysis
**INSTRUCTIONS:**
1. Ensure you have added the original `reviews-to-classify` dataset (for `reviews_to_classify.csv`).
2. **Upload** your `classified_reviews.csv` file directly into `/kaggle/working/` (the right-hand sidebar).
3. Turn on the **T4 GPU** accelerator in the notebook settings.
4. Run all cells!

In [ ]:
!pip install transformers sentencepiece accelerate vaderSentiment pandas tqdm -q

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm.auto import tqdm
import os
from IPython.display import FileLink, display

In [ ]:
import glob
import pandas as pd

# Use glob to find the files no matter how you uploaded them to Kaggle!
reviews_path = glob.glob('/kaggle/input/**/reviews_to_classify.csv', recursive=True)
class_path = glob.glob('/kaggle/input/**/classified_reviews.csv', recursive=True)
if not class_path:
    class_path = glob.glob('/kaggle/working/**/classified_reviews.csv', recursive=True)

if not reviews_path:
    raise FileNotFoundError("Could not find reviews_to_classify.csv! Please make sure to click '+ Add Input' and attach your original dataset.")
if not class_path:
    raise FileNotFoundError("Could not find classified_reviews.csv! Please make sure it is uploaded in the Data section.")

print(f'Found texts at: {reviews_path[0]}')
print(f'Found classifications at: {class_path[0]}')

df_texts = pd.read_csv(reviews_path[0])
df_class = pd.read_csv(class_path[0])

df = pd.merge(df_class, df_texts, on='InspectionID', how='inner')
print(f'Merged {len(df)} reviews for precision processing.')

In [ ]:
def map_to_taxonomy(category):
    c = str(category).lower()
    if any(x in c for x in ['fraud', 'scam', 'never received', 'no delivery', 'não recebido', 'not received', 'empty']): return 'Critical Failure', 5.0
    if any(x in c for x in ['damaged', 'defective', 'shattered', 'quebrado', 'broken']): return 'Major Product Issue', 4.0
    if any(x in c for x in ['wrong item', 'missing parts', 'color', 'size', 'errado', 'mismatch', 'no fit']): return 'Fulfillment Error', 3.5
    if any(x in c for x in ['late', 'delay', 'atraso', 'delayed']): return 'Logistics Issue', 3.0
    if any(x in c for x in ['poor quality', 'flimsy', 'ruim', 'quality']): return 'Quality Issue', 2.5
    if any(x in c for x in ['rude', 'communication', 'service', 'resposta']): return 'Service Defect', 2.0
    return 'Not a Defect', 0.0

print('Applying 7-Tier Supply Chain Taxonomy...')
df[['TaxonomyCategory', 'BaselineSeverity']] = df['DefectCategory'].apply(lambda x: pd.Series(map_to_taxonomy(x)))
print('\nTaxonomy Distribution:')
print(df['TaxonomyCategory'].value_counts())

In [ ]:
print('Loading translation model (Helsinki-NLP/opus-mt-pt-en) on GPU...')
device = 0 if torch.cuda.is_available() else -1
translator = pipeline('translation', model='Helsinki-NLP/opus-mt-pt-en', device=device)

texts = df['ReviewText'].fillna('').astype(str).tolist()
translated_texts = []

batch_size = 32
for i in tqdm(range(0, len(texts), batch_size), desc='Translating to English'):
    batch = texts[i:i+batch_size]
    # Max length bounded to prevent errors on huge reviews
    batch = [t[:400] for t in batch]
    results = translator(batch, max_length=512)
    translated_texts.extend([r['translation_text'] for r in results])

df['TranslatedText'] = translated_texts

In [ ]:
print('Applying VADER Sentiment Analysis...')
analyzer = SentimentIntensityAnalyzer()
compound_scores = []
for text in tqdm(df['TranslatedText'], desc='Scoring'):
    score = analyzer.polarity_scores(text)['compound']
    compound_scores.append(score)
    
df['VaderCompound'] = compound_scores

# Math: Baseline severity increases if sentiment is negative. Decreases if positive.
# Since compound is -1 to 1, we do: Baseline - (Compound * 0.5)
# Example: Compound -1.0 (very angry) adds +0.5 to severity.
df['FluidSeverity'] = df['BaselineSeverity'] - (df['VaderCompound'] * 0.5)
df['FluidSeverity'] = np.where(df['TaxonomyCategory'] == 'Not a Defect', 0.0, df['FluidSeverity'])
df['FluidSeverity'] = df['FluidSeverity'].clip(0.0, 5.0).round(2)

print('\nSample of calculated severities:')
print(df[['TaxonomyCategory', 'BaselineSeverity', 'VaderCompound', 'FluidSeverity']].head(10))

In [ ]:
output_file = 'vader_classified_reviews.csv'
# Keep only necessary columns to avoid bloat
out_df = df[['InspectionID', 'TaxonomyCategory', 'FluidSeverity', 'TranslatedText', 'VaderCompound']]
out_df = out_df.rename(columns={'TaxonomyCategory': 'DefectCategory', 'FluidSeverity': 'SeverityWeight'})
out_df.to_csv(output_file, index=False)
print(f'\nDone! File saved to {os.path.abspath(output_file)}')

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    print('Click the link below to download your file:')
    display(FileLink(output_file))
else:
    try:
        from google.colab import files
        files.download(output_file)
    except:
        pass